In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [ ]:
# 1. Image Data Generator (Membaca gambar & Melakukan Augmentasi Otomatis)

In [ ]:
print("1. Menyiapkan Data Generators...")
# augmentasi
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True
)

In [ ]:
# data validasi dan testing
test_val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [ ]:
# Load Data
train_generator = train_datagen.flow_from_directory(
    '/content/split_dataset/train',
    target_size=(224, 224), batch_size=32, class_mode='categorical'
)

val_generator = test_val_datagen.flow_from_directory(
    '/content/split_dataset/val',
    target_size=(224, 224), batch_size=32, class_mode='categorical'
)

test_generator = test_val_datagen.flow_from_directory(
    '/content/split_dataset/test',
    target_size=(224, 224), batch_size=32, class_mode='categorical',
    shuffle=False
)

print("\nUrutan Kelas:", train_generator.class_indices)
class_names = list(train_generator.class_indices.keys())

In [ ]:
print("\n2. Mengunduh dan Membangun Model MobileNetV2...")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x) # Cegah Overfitting
predictions = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# 3. Training

In [ ]:
print("\n3. Mulai Training Model...")
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    ModelCheckpoint("best_vision_model.h5", monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
# 4. Grafik training

In [ ]:
print("\n4. Membuat Grafik Evaluasi Training...")
plt.figure(figsize=(12, 4))

# Grafik Akurasi
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Grafik Akurasi')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Grafik Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Grafik Loss (Kesalahan)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5. TESTING & MATRIKS EVALUASI TAHAP AKHIR

In [ ]:
print("\n5. Menguji Model pada Data Testing (Data yang belum pernah dilihat model)...")
test_loss, test_acc = model.evaluate(test_generator)
print(f"Akurasi pada Test Data: {test_acc*100:.2f}%")

print("\nMembuat Confusion Matrix dan Classification Report...")
# Prediksi data test
Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

In [ ]:
# 5A. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Label Asli')
plt.xlabel('Label Prediksi Model')
plt.show()

In [ ]:
# 5B. Classification Report
print("\nCLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=class_names))
print("\nSELESAI! File 'best_vision_model.h5' telah disimpan dan dievaluasi.")